# 02. RQ2, comparing the methods

> Reference notebook. Every number below is produced by the same functions that
> write the paper's tables (`paper_resources/scripts/`), so this is a readable
> view of the pipeline rather than a second implementation. Regenerate the
> `.tex` files with `python scripts/run_all.py`.


**Question.** With the design factors from RQ1 fixed, which methods lead, and
does combining separately selected factors compose?

Backs Table 1 (shared-panel method comparison), Table A7 (promoted shortlist),
Table A8 (full follow-up leaderboard), Table A9 (mixing effects), and Table A10
(follow-up tests).


In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in Path.cwd().resolve().parents if (p / "paper_resources").is_dir())
sys.path.insert(0, str(REPO_ROOT / "paper_resources" / "scripts"))

import common as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
# 4 significant digits rather than 4 decimals, so that a p-value of 5.6e-15 stays
# readable next to an effect size of 0.94.
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

cfg = C.load_config()
BENCHMARKS = cfg["analysis"]["benchmarks"]
ALPHA = cfg["analysis"]["alpha"]
print(f"repo root : {REPO_ROOT}")
print(f"benchmarks: {BENCHMARKS}")
print(f"alpha     : {ALPHA}")

repo root : /artifact/moe-study-analysis
benchmarks: ['Belebele', 'FLORES', 'SIB200', 'Taxi1500']
alpha     : 0.05


## Six methods on a shared rank panel

This is the largest set of methods comparable without letting differences in
grid coverage decide the outcome, namely the six run at *every* rank from 2
to 64.


In [2]:
family = pd.read_csv(C.output_dir(cfg, "data") / "tab2_rq2_family_comparison.csv")
family

,family,method,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
0,moelora,MOELoRA,2.037,3.324,2.815,2.556,2.683
1,mola,AdaMoE,2.62,2.796,2.63,2.833,2.72
2,hydralora,HydraLoRA,3.361,3.12,2.741,3.792,3.253
3,mtllora,MTL-LoRA,3.778,2.972,4.176,3.75,3.669
4,lora,LoRA,3.352,2.88,4.759,3.931,3.73
5,hmora,HMoRA,5.852,5.907,3.88,4.139,4.944


MOELoRA and AdaMoE take the best overall positions, ahead of HydraLoRA. The
per-benchmark leaders differ, which is the first hint of the task-dependence
that RQ4 makes central.


## Does mixing selected factors compose?

The search selected a good rank and a good expert count *separately*. The
mixing analysis asks whether combining them beats the better parent. If
one-factor trends composed, every mixed descendant would win.


In [3]:
mixing = pd.read_csv(C.output_dir(cfg, "data") / "tab2_rq2_mixing_effects.csv")
mixing

,architecture,configured_rank,parent,mixed_child,parent_macro_rank,mixed_macro_rank,delta_Belebele,delta_FLORES,delta_SIB200,delta_Taxi1500,delta_macro
0,MOELoRA,2,moelora_tr16_e8_k2,moelora_tr32_e16_k8,13.8,7.479,1.444,9.444,9.722,4.667,6.319
1,MOELoRA,8,moelora_tr64_e8_k2,moelora_tr256_e32_k8,12.47,10.17,11.39,2.222,5.667,-10.08,2.299
2,AdaMoE,2,mola_tr16_e8_k6,mola_tr64_e32_k4,9.236,11.63,-4.167,-5.222,-5.944,5.75,-2.396
3,AdaMoE,8,mola_tr64_e8_k6,mola_tr256_e32_k4,17.96,12.02,-1.778,4.444,7.5,13.58,5.938
4,HydraLoRA,2,hydralora_r2,hydralora_r2_heads32,7.5,11.72,1.222,-1.611,-5.389,-11.08,-4.215
5,MTL-LoRA,8,mtllora_r8,mtllora_r8_nup16,10.7,7.285,1.056,9.556,1.889,1.167,3.417


In [4]:
all_desc = pd.read_csv(
    C.output_dir(cfg, "data") / "tab2_rq2_mixing_effects_all_descendants.csv"
)
all_desc.head(20)

,architecture,configured_rank,parent,mixed_child,parent_macro_rank,mixed_macro_rank,delta_Belebele,delta_FLORES,delta_SIB200,delta_Taxi1500,delta_macro
0,MOELoRA,2,moelora_tr16_e8_k2,moelora_tr32_e16_k8,13.8,7.479,1.444,9.444,9.722,4.667,6.319
1,MOELoRA,2,moelora_tr16_e8_k2,moelora_tr64_e32_k4,13.8,9.382,-1.111,6.056,10.06,2.667,4.417
2,MOELoRA,2,moelora_tr16_e8_k2,moelora_tr64_e32_k8,13.8,8.181,3.722,5.167,6.833,6.75,5.618
3,MOELoRA,8,moelora_tr64_e8_k2,moelora_tr128_e16_k8,12.47,11.49,10.22,-3.278,7.556,-10.58,0.9792
4,MOELoRA,8,moelora_tr64_e8_k2,moelora_tr256_e32_k4,12.47,12.31,9.333,-6.278,7.833,-10.25,0.1597
5,MOELoRA,8,moelora_tr64_e8_k2,moelora_tr256_e32_k8,12.47,10.17,11.39,2.222,5.667,-10.08,2.299
6,AdaMoE,2,mola_tr16_e8_k6,mola_tr32_e16_k1,9.236,13.46,-2.889,-12.89,-1.778,0.6667,-4.222
7,AdaMoE,2,mola_tr16_e8_k6,mola_tr32_e16_k8,9.236,13.96,-4.944,-8.444,1.667,-7.167,-4.722
8,AdaMoE,2,mola_tr16_e8_k6,mola_tr64_e32_k4,9.236,11.63,-4.167,-5.222,-5.944,5.75,-2.396
9,AdaMoE,8,mola_tr64_e8_k6,mola_tr128_e16_k1,17.96,12.87,2.333,-4.111,14.11,8,5.083


The best mixed descendant fails to beat its preserved parent in two of six
prespecified groups. One-factor trends do not reliably compose, which is what
motivates validating in the regime you intend to deploy.


## The longer follow-up

Twenty-two configurations retrained at the longer budget on ten languages. This
stage produces the shortlist of five.


In [5]:
followup = C.load_primary(C.results_path(cfg, "mixed_followup_xlsx"), cfg)
followup_ranking, per_benchmark = C.followup_ranking(followup, BENCHMARKS)
followup_ranking.head(10).round(2)

score_family,overall_rank,approach,Belebele,FLORES,SIB200,Taxi1500,macro_avg_rank
run_id,,,,,,,
mtllora_r8_nup16,1,mtllora,7.11,10.28,4.33,7.42,7.28
moelora_tr32_e16_k8,2,moelora,6.44,4.94,11.61,6.92,7.48
hydralora_r2,3,hydralora,10.83,9.89,4.11,5.17,7.5
moelora_tr64_e32_k8,4,moelora,4.17,9.22,14.5,4.83,8.18
mola_tr16_e8_k6,5,mola,10.83,4.33,8.94,12.83,9.24
moelora_tr64_e32_k4,6,moelora,9,8.33,11.28,8.92,9.38
moelora_tr256_e32_k8,7,moelora,7.06,4.06,16,13.58,10.17
mtllora_r8,8,mtllora,8.17,19.83,6.22,8.58,10.7
moelora_tr128_e16_k8,9,moelora,8.22,9.56,14.11,14.08,11.49


In [6]:
omnibus, pairwise = C.followup_tests(followup, BENCHMARKS, cfg["rq2"]["finalists"])
display(omnibus)
pairwise

,benchmark,n_units,friedman_q,kendalls_w,p_raw,p_holm
0,Belebele,9,77.47,0.4099,2.125e-08,4.25e-08
1,FLORES,18,193,0.5106,8.6e-30,3.44e-29
2,SIB200,9,115.7,0.6124,4.277e-15,1.283e-14
3,Taxi1500,6,76.13,0.6042,3.54e-08,4.25e-08


,benchmark,config_a,config_b,median_difference,wilcoxon_w,p_raw,rank_biserial,p_holm
0,Belebele,mtllora_r8_nup16,moelora_tr32_e16_k8,-0.004444,18,0.6523,-0.2,1
1,Belebele,mtllora_r8_nup16,hydralora_r2,0.002222,8.5,0.2109,0.5278,1
2,Belebele,moelora_tr32_e16_k8,hydralora_r2,0.01111,8,0.09766,0.6444,0.7812
3,FLORES,mtllora_r8_nup16,moelora_tr32_e16_k8,-0.001113,15,0.001045,-0.8246,0.0115
4,FLORES,mtllora_r8_nup16,hydralora_r2,0.0001889,76,0.7019,0.1111,1
5,FLORES,moelora_tr32_e16_k8,hydralora_r2,0.0015,8,0.0001907,0.9064,0.002289
6,SIB200,mtllora_r8_nup16,moelora_tr32_e16_k8,0.03922,3,0.01953,0.8667,0.1758
7,SIB200,mtllora_r8_nup16,hydralora_r2,0,14,1,0,1
8,SIB200,moelora_tr32_e16_k8,hydralora_r2,-0.03431,1,0.007812,-0.9556,0.07812
9,Taxi1500,mtllora_r8_nup16,moelora_tr32_e16_k8,0.01351,9,0.8438,0.1429,1


Only FLORES separates the top three. Because the band is narrow and just one
benchmark distinguishes its members, all five configurations are carried
forward rather than the single best.

**Reading discipline.** This stage used the same observations to select and to
compare, so these orderings describe the search. They are not confirmatory
evidence, and the paper says so explicitly.
